
DataLake (Deltalake) + Lakehouse (Deltatables) - using Delta format (parquet+snappy+delta log)

Delta Lake is an open-source storage framework that brings reliability, ACID transactions, and performance to data lakes. It sits on top of Parquet files and is most commonly used with Apache Spark and Databricks.

Delta Lake & Deltalakhouse is the Core/Analytical storage layer behind Bronze–Silver–Gold (medallion) architectures.

Creating our first Delta Lake table
Delta is the default file and table format using Databricks.

Delta support parquet data only

iceberg support parquet / orc/ avro

In [0]:
spark.sql("create catalog if not exists lakehousecat1")
spark.sql("create schema if not exists lakehousecat1.deltadb;")
spark.sql("create volume if not exists lakehousecat1.deltadb.datalake;")
spark.sql(f"""create volume if not exists lakehousecat1.deltadb.delta1;""")

1. Write data into delta file (Datalake) and table (Lakehouse)

- How to migrate csv to delta format
- Difference between Delta and Parquet
- How to create Datalake & Lakehouse

In [0]:

#1. How to migrate csv to delta format (Delta Lake creation)
df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/Master_City_List.csv",header="True")
df.write.format("delta").mode("overwrite").save("/Volumes/lakehousecat1/deltadb/datalake")

#%fs head /Volumes/lakehousecat1/deltadb/datalake/_delta_log/00000000000000000000.json
#%fs head /Volumes/lakehousecat1/deltadb/datalake/_delta_log/00000000000000000000.crc
#%fs head  /Volumes/lakehousecat1/deltadb/datalake/part-00000-e911283b-ea62-4755-8041-5c4f99cf07ec.c000.snappy.parquet

#2. Difference between Delta and Parquet
df.write.format("parquet").mode("overwrite").save("/Volumes/lakehousecat1/deltadb/datalake/targetparquet") #writing normal data into parquet(datalake)
df.write.mode("overwrite").save("/Volumes/lakehousecat1/deltadb/datalake/targetdelta") #Databricks default format is delta(parquet)

#3. How to create Delta Lakehouse
spark.sql("drop table if exists lakehousecat1.deltadb.drugstbl")
df.write.saveAsTable("lakehousecat1.deltadb.drugstbl",mode="overwrite")  #writing normal data from deltalakehouse(lakehouse)
#behind it stores the data in deltafile format in the s3 bucket (location is hidden for us in databricks free edition)

In [0]:

%sql
select * from lakehousecat1.deltadb.drugstbl;
explain select * from lakehousecat1.deltadb.drugstbl;
--under the hood data is stored in S3

We can have schema evolution performed.
We can add Schema evolution feature just by adding the below option in Delta tables.

In [0]:
df.write.option("mergeSchema","True").saveAsTable("lakehousecat1.deltadb.drugstbl",mode="overwrite")

2. DML Operations in Delta Tables & Files

We are overcoming the WORM (Write Once Read Many) limitation in Cloud S3/GCS/ADLS or in Distributed storage layers like HDFS

Delta file/table supports WMRM(Write Manay Read Many) operations, using DMLs such as INSERT/DELETE/UPDATE/MERGE

In [0]:

%sql
--DDL is supportive (we will do more of these further)
create or replace table lakehousecat1.deltadb.sampletable (id int , name string) using delta ;

insert into lakehousecat1.deltadb.sampletable values (1,"vicky"); --Though the data is stored internally in delta file, we can't see the data in delta format in databricks serverless
insert into lakehousecat1.deltadb.sampletable values (2,"virat"); 

update lakehousecat1.deltadb.sampletable set id  = 3 where name = "virat";
select distinct * from lakehousecat1.deltadb.sampletable; 

--DML - merge is possible in the delta tables/files
describe history  lakehousecat1.deltadb.sampletable

In [0]:

%sql
insert into lakehousecat1.deltadb.sampletable values(2,'IZUSER');
insert into lakehousecat1.deltadb.sampletable values(2,'IZUSER');
insert into lakehousecat1.deltadb.sampletable values(2,'IZUSER');
insert into lakehousecat1.deltadb.sampletable values(2,'IZUSER');
insert into lakehousecat1.deltadb.sampletable values(2,'IZUSER');


insert into lakehousecat1.deltadb.sampletable values(2,'IZUSER'),(2,'IZUSER'),(2,'IZUSER'),(2,'IZUSER'),(2,'IZUSER'),(2,'IZUSER'),(2,'IZUSER'),(2,'IZUSER'),(2,'IZUSER'),(2,'IZUSER');

-- every commit / successful operation creates a new version
-- every changes in the table creates a new version

-- how many version we can have it table (no limit) - no limit based on on version num  based on ts we have some soft limit 



-- checkpointing / optimize / vaccum 

In [0]:
%sql
describe history  lakehousecat1.deltadb.sampletable;

In [0]:
%sql


-- version as of or timestamp 
select * from lakehousecat1.deltadb.sampletable version as of 2;
select * from lakehousecat1.deltadb.sampletable timestamp as of '2026-08-01T07:30:00.000+00:00';
describe formatted lakehousecat1.deltadb.sampletable;

In [0]:
%sql
use lakehousecat1.deltadb;
desc history lakehousecat1.deltadb.sampletable

In [0]:
%sql
-- dql is supported 
select * from lakehousecat1.deltadb.sampletable where id =1;

a. Table Update

In [0]:
%sql
update 
sampletable
set id =id +1 where id =1;



--default latest version will be shown
select * from lakehousecat1.deltadb.sampletable where id =1;
select * from lakehousecat1.deltadb.sampletable  version as of 1 where id =1;

b. Table Delete

In [0]:
%sql
delete from lakehousecat1.deltadb.sampletable
where id =1;

desc history lakehousecat1.deltadb.sampletable;

c. File DML (Update/Delete)

We don't do file DML usually, we are doing here just for learning about

file also can be undergone with limited DML operation

we need to learn about how the background delta operation is happening when i do DML

In [0]:
# read delta files and create DF 
spark.read.format('delta').option("versionAsOf",0).load('/Volumes/lakehousecat1/deltadb/datalake').show()

In [0]:
#DML on Files: How to update delta files (Not used very frequently)
from delta.tables import DeltaTable
deltafile = DeltaTable.forPath(spark, "`/Volumes/lakehousecat1/deltadb/datalake`")
deltafile.update("latitude=12.9716", { "city_name": "tvl" } )


In [0]:

%sql

update delta.`/Volumes/lakehousecat1/deltadb/datalake` 
set city_name="tvl"
where latitude=12.9716;

d. Merge Operation

In [0]:

%sql
select count(*) from  lakehousecat1.deltadb.sampletable

In [0]:
%sql
--CTAAS
CREATE OR REPLACE TABLE lakehousecat1.deltadb.sampletablemerge as select * from lakehousecat1.deltadb.sampletable where id =2;
select count(*) from  lakehousecat1.deltadb.sampletablemerge; --16
select * from  lakehousecat1.deltadb.sampletable; --17

select 100-99


In [0]:
%sql
--merge syntax
--merge into targetable using sourcetable on condition_to_join
--when matched then update
--when not matched then insert
--when not matched by source then delete (some additional data in the target should be deleted, which is already deleted in source)
--Delta table support merge operation for (insert/update/delete)

merge into lakehousecat1.deltadb.sampletablemerge t
using lakehousecat1.deltadb.sampletable s on s.id = t.id 
when matched then delete 
when not matched then insert *

In [0]:
%sql
--What if the source got some data removed and which is present in the target still (we can leave it or delete)
insert into lakehousecat1.deltadb.sampletablemerge select 99999999,name
from lakehousecat1.deltadb.sampletablemerge limit 1;

In [0]:
%sql
--Target table contains excessive data
select count(*) from  lakehousecat1.deltadb.sampletablemerge; --16
select count(*) from  lakehousecat1.deltadb.sampletable; --17

In [0]:

%sql
--Delta table support merge operation for (delete)
--1 deleted (which is not present in the source (source system deleted it already, hence target also has to delete))

merge into lakehousecat1.deltadb.sampletablemerge t
using lakehousecat1.deltadb.sampletable s on s.id = t.id 
when matched then delete 
when not matched then insert (id,name) values (89,"koing")

In [0]:

#Few points to consider regarding merge...
#1. Merge can be only applied on tables in Databricks delta
#2. Merge operation using spark with (library delta.tables.DeltaTable) DSL (not by using SQL) - SQL is better to use

from delta.tables import DeltaTable
print(spark.read.table("lakehousecat1.deltadb.sampletable").count())
print(spark.read.table(" lakehousecat1.deltadb.sampletablemerge").count())
tgt = DeltaTable.forName(spark, "lakehousecat1.deltadb.sampletablemerge")
src = spark.table("lakehousecat1.deltadb.sampletable")
(
    tgt.alias("t").merge(src.alias("src"), "t.id = src.id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
)

src = spark.table("lakehousecat1.deltadb.sampletable")
(
    tgt.alias("tgt")
    .merge(
        src.alias("src"),
        "tgt.id = src.id"
    )
    .whenMatchedUpdate(set={
        "id": "src.id",
        "name": "src.name"
    })
    .whenNotMatchedInsert(values={
        "id": "src.id",
        "name": "src.name"
    })
    .whenNotMatchedBySourceDelete()
    .execute() )

3. Additional Operations on Deltalake & Deltatables

a. History & Versioning
History returns one row per commit/version and tells you what changed, when, and how.

In [0]:
%sql
DESC HISTORY lakehousecat1.deltadb.sampletablemerge

Version as of will reads the snapshot of drugstbl_merge at version 4 and Ignores all changes made in versions 5, 6, … current

In [0]:
%sql
--select * from (select * from deltadb.drugs version as of 2) where uniqueid=163740;
SELECT count(*) FROM lakehousecat1.deltadb.sampletablemerge VERSION AS OF 2;--Behind the scene, databricks sql engine with the help of deltaengine (it will read the log and the respective data and produce the output)
     

b. Time Travel
Timestamp as of Reads the table as it existed at that exact timestamp and Any commits after the given timestamp is ignored

In [0]:

%sql
SELECT * FROM lakehousecat1.deltadb.sampletablemerge TIMESTAMP AS OF '2026-08-01T16:24:44.000+00:00';

c. Vaccum
VACUUM in Delta Lake removes old, unused files to free up storage, default retention hours is 168. These files come from operations like DELETE, UPDATE, or MERGE and are kept temporarily so time-travel queries can work.

Before VACUUM
Active + deleted parquet files exist

After VACUUM
Only ACTIVE parquet files remains and delete Old parquet files (from UPDATE/MERGE/DELETE)
Logs remain intact
Time travel beyond retention becomes impossible

In [0]:

%sql
--use lakehousecat1.deltadb;
alter table lakehousecat1.deltadb.sampletablemerge SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = '24 hours');
VACUUM lakehousecat1.deltadb.sampletablemerge;--default value in databricks, we can reduce or increase this(but in serverless it is not possible to reduce)

AI Suggested feature
Setting the Delta table property 'delta.deletedFileRetentionDuration' to less than the default (1 week) is generally not recommended for production environments. Lowering the retention duration can lead to data loss if you need to time travel or restore data, as older files may be deleted sooner than expected. The default of 168 hours (1 week) is chosen to balance storage costs and safety for production workloads. Only reduce this value if you fully understand the risks and have a strong operational reason to do so

In [0]:
%sql
--default 168 hours (1 week), less than 1 week will not work in serverless for performance and session state reason
VACUUM lakehousecat1.deltadb.sampletablemerge RETAIN 168 HOURS;

d. ACID Transactions
Delta Lake supports ACID transactions under the hood via a transaction log.

ACID	In Databricks
Atomicity	Every transactions are Individual Transactions / All or nothing
Consistency	Schema + constraints
Isolation	Using Version/Time/restore we can isolate transactions, we can't use TCL (commit/rollback)
Durability	Every transaction Always hit the disk (durable), but can be controlled by Transaction log

In [0]:
%sql
use lakehousecat1.deltadb;

CREATE OR REPLACE TABLE acid_demo_txn (
  id INT,
  amount INT
) USING DELTA;



select * from acid_demo_txn;

In [0]:

%sql
--Atomiocity  -- all or nothing
--All or nothing (Atomic/Individual Transaction) - help us make a transaction complete or fail, hence no partial data
INSERT INTO acid_demo_txn VALUES--One atomic (all or nothing) transaction is inserting 3 rows
(1, 100),
(2, 200),
(3, 300);
--(3, '300');
--This will make the entire transaction failed (all or nothing)


select * from acid_demo_txn;

In [0]:

%sql
--Atomicity (Individual transaction that doesn't affect the other)
UPDATE acid_demo_txn SET amount = amount + 100 WHERE id = 1;--individual/atomic
--The above statement is atomic, hence the below statement take amount as 200 and added 200 more
UPDATE acid_demo_txn SET amount = amount + 200 WHERE id = 1;--individual/atomic
describe history acid_demo_txn;
--START TRANSACTION; UPDATE acid_demo_txn SET amount = amount + 100 WHERE id = 1; commit;
--START TRANSACTION; UPDATE acid_demo_txn SET amount = amount + 200 WHERE id = 1; commit;

In [0]:
%sql
select * from acid_demo_txn where id=1;

In [0]:
%sql
--Apply constraint for maintaining consistancy
--We can apply in databricks deltatable, 2 types of constraints (check and not null), 
-- in other DBs we can use primary key, foreign key and unique constraints also..
--ALTER TABLE acid_demo_txn ADD CONSTRAINT positive_amount CHECK (amount > 0);
INSERT INTO acid_demo_txn VALUES (4, 100);--Atomicity and consistancy
--INSERT INTO acid_demo_txn VALUES (5, -100);--Atomicity and consistancy


--Only consistant data is loaded
select * from acid_demo_txn;

In [0]:
%sql
--Isolation (We can achieve using timetravel (restore operation (no rollback)))
--START TRANSACTION; SAVEPOINT before_delete; DELETE FROM employees WHERE employee_id = 129; ROLLBACK TO before_delete;
--Notebook1 (We can see the data in notebook1)
UPDATE acid_demo_txn SET amount = 999 WHERE id = 2;--This update will write the data in the disk with version added
--Notebook2 (We can see the data in notebook2 )
--use lakehousecat1.deltadb;
--select * from (select * from acid_demo_txn version as of 14) where id=2;--serializable read (after the data successfully committed)

In [0]:

%sql
select * from acid_demo_txn;

In [0]:
%sql
--something like savepoint+rollback (but not really a rollback (TCL is not available in Databricks in the name of commit, rollback, savepoint))
restore table acid_demo_txn to version as of 1;
     

In [0]:
%sql
--Durability (Despite of terminate and starting back the serverless, data still survives durably)
INSERT INTO acid_demo_txn VALUES (5, 500);

In [0]:

%sql
use lakehousecat1.deltadb;
select * from acid_demo_txn;

e. Transactions Control (TCL cannot be achieved using commit/rollback/savepoint)
Bigdata ecosystems such as spark/databricks/delta are not Transaction in nature, hence it will not support TCL directly, but can be achieved using version/timetravel/restore

In [0]:

%sql
--select count(1) from deltadb.drugs where date>'2012-02-28';
--4329
--Equivalent to delete and commit (with version (savepoint))
delete from acid_demo_txn where id =1;


select count(1) from acid_demo_txn;



--Equivalent to restore to a version
--Equivalent to Rollback to a savepoint
RESTORE TABLE acid_demo_txn TO VERSION AS OF 2;


--We can restore to any older/later version (unlit 168 hours/vacumm period)
RESTORE TABLE acid_demo_txn TO VERSION AS OF 1;

In [0]:

%sql
select count(1) from drugstbl;

In [0]:
%sql
--We can restore to any older/later version (unlit 168 hours/vacumm period)
RESTORE TABLE drugstbl TO VERSION AS OF 4;

In [0]:
%sql
describe history drugstbl;

In [0]:

%sql
select count(1) from drugstbl;
     

Drop and undrop

In [0]:

%sql

use lakehousecat1.deltadb ;

create table emp_drop(eid int,ename string);

insert into emp_drop values(1,'a'),(2,'b'),(3,'c');

insert into emp_drop values(4,"raja");

select * from emp_drop;
    
describe history emp_drop;


drop table emp_drop;
     

In [0]:
%sql

show tables in  lakehousecat1.deltadb;

show tables dropped in  lakehousecat1.deltadb;


undrop table emp_drop;

select * from emp_drop;

describe history emp_drop;  -- 7 days

-- we took the backup of underlying dir (delta_log , all.parq)
-- emp  ---> emp.bkp 

-- create table cat.sche.emp using delta location 's3://buck/emp.bkp'; 

In [0]:
# create dataframe with emp id,name age
df = spark.createDataFrame([(1,'krish',20),(2,'raja',30),(3,'abc',40)],['id','name','age'])

df.display()
# create dataframe with dept id, dept name
df.write.format("delta").mode("overwrite").save("/Volumes/lakehousecat1/deltadb/delta1/dv_demo1")

In [0]:
spark.sql("DESCRIBE HISTORY delta.`/Volumes/lakehousecat1/deltadb/delta1/dv_demo1`").display()

In [0]:
%sql
desc history  delta.`/Volumes/lakehousecat1/deltadb/delta1/dv_demo1`;

show tblproperties delta.`/Volumes/lakehousecat1/deltadb/delta1/dv_demo1`;

-- enable / disable the dv feature 
alter table delta.`/Volumes/lakehousecat1/deltadb/delta1/dv_demo1` 
set tblproperties (delta.enableDeletionVectors = false);


update delta.`/Volumes/lakehousecat1/deltadb/delta1/dv_demo1` set name = 'sachin' where id = 2;

delete from  delta.`/Volumes/lakehousecat1/deltadb/delta1/dv_demo1` where id = 2;

In [0]:
df = spark.createDataFrame([(1,'krish',20),(2,'raja',30),(3,'abc',40)],['id','name','age'])

df.display()
# create dataframe with dept id, dept name
df.write.format("delta").mode("overwrite").save("/Volumes/lakehousecat1/deltadb/delta1/dv_demo2")


In [0]:
%sql
describe history delta.`/Volumes/lakehousecat1/deltadb/delta1/dv_demo2`;


show tblproperties delta.`/Volumes/lakehousecat1/deltadb/delta1/dv_demo2`;


update delta.`/Volumes/lakehousecat1/deltadb/delta1/dv_demo2` set name = 'sachin' where id = 2;

delete from  delta.`/Volumes/lakehousecat1/deltadb/delta1/dv_demo2` where id = 2;

In [0]:

%sql

desc history delta.`/Volumes/lakehousecat1/deltadb/delta1/dv_demo2`;
vacuum delta.`/Volumes/lakehousecat1/deltadb/delta1/dv_demo2`;